In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json

url = "https://www.imdb.com/chart/toptv/"

h = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.5"
}

response = requests.get(url, headers=h)
soup = BeautifulSoup(response.text, "html.parser")

#finding hidden json data
script_tag = soup.find("script", id="__NEXT_DATA__")

if not script_tag:
    print("Could not find the JSON data. IMDb might have changed their page structure.")
    exit()

#change json to dictinary
# change json to dictionary
data = json.loads(script_tag.string)

# correct path to list of shows
items = data['props']['pageProps']['pageData']['chartTitles']['edges']

ranks = []
titles = []
years = []
ratings = []

for i, item in enumerate(items, start=1):
    node = item['node']

    ranks.append(i)
    titles.append(node['titleText']['text'])
    years.append(node['releaseYear']['year'])
    ratings.append(node['ratingsSummary']['aggregateRating'])

df = pd.DataFrame({
    "Rank": ranks,
    "Title": titles,
    "Year": years,
    "Rating": ratings
})

df.set_index("Rank", inplace=True)
df.to_json("top_250_tv_shows.json", orient="records", indent=4)
df.to_csv("top_250_tv_shows.csv", index=True)
print(f"Successfully scraped {len(df)} records!")


Successfully scraped 250 records!


In [6]:
from google.colab import files

files.download("top_250_tv_shows.csv")
files.download("top_250_tv_shows.json")


FileNotFoundError: Cannot find file: top_250_tv_shows.csv